# DiffCAM with a feature extractor and no classifier

**Weeks:** 36 · **Core question:** Why does DiffCAM not need a classifier head?

**Prerequisites:** complete preceding notebooks in the curriculum. Each lesson targets 45–90 minutes including exercises.

## Concept and motivation

The core DiffCAM direction uses feature embeddings and reference examples; no predicted class score is needed. This lesson uses a **handcrafted, fixed feature extractor** as a transparent proxy. It is *not* a trained DINO or MoCo model. Later, replace the extractor with a legitimate pretrained self-supervised encoder using documented preprocessing.

**Key vocabulary:** representation learning, self-supervised pretraining, encoder, classifier-free explanation.

**Learning goal:** Explain why absence of a classification score is an algorithmic property, not proof of faithfulness.

> All images and patient-like examples in this repository are synthetic teaching artifacts, not clinical data.


## Prepare the environment
Run this cell first. All core examples are offline and reproducible.

In [ ]:
from pathlib import Path
import sys
# Works when launched in the repository, even from a nested notebook directory.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'diffcam_edu').exists()), None)
if root is None:
    raise RuntimeError('Open Jupyter from the extracted diffcam_curriculum folder, or set working directory to it.')
sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)


## Guided experiment
Execute the code cells in order. Before running each cell, predict what it will show; afterward, explain what changed and why.

### Experiment 1 of 3

In [ ]:
from scipy.ndimage import gaussian_filter,sobel
from diffcam_edu import make_cell_images
from diffcam_edu.math import diffcam_weights,diffcam_map,normalize_map
x,y,m=make_cell_images(80,seed=63)
def frozen_encoder(image):
    return np.stack([image,gaussian_filter(image,1),np.abs(sobel(image,0)),np.abs(sobel(image,1))])
feature_maps=np.stack([frozen_encoder(a) for a in x]); embeddings=feature_maps.mean(axis=(-2,-1))
print('Frozen maps:',feature_maps.shape,'Embeddings:',embeddings.shape)

### Experiment 2 of 3

In [ ]:
j=int(np.flatnonzero(y==1)[0]);w=diffcam_weights(embeddings[j],embeddings[y==0],ridge=.2)
h=normalize_map(diffcam_map(feature_maps[j],w))
plt.figure(figsize=(4,4));plt.imshow(x[j],cmap='gray');plt.imshow(h,cmap='inferno',alpha=.5)
plt.title('Classifier-free feature-difference map');plt.axis('off');plt.show()

### Experiment 3 of 3

In [ ]:
print('Weights computed using only image-derived embeddings:',w)
assert len(w)==feature_maps.shape[1]

## Exercises — complete independently
1. Replace the handcrafted feature extractor with PCA applied to local image patches. What must be done to preserve spatial arrangement?
2. Why does classifier-free explanation still depend strongly on encoder quality?

## Check your understanding

**Exit ticket:** Explain why absence of a classification score is an algorithmic property, not proof of faithfulness.

Write a short answer and save a screenshot or plot from your experiment.

## Further reading
- [DiffCAM (CVPR 2025)](https://openaccess.thecvf.com/content/CVPR2025/html/Li_DiffCAM_Data-Driven_Saliency_Maps_by_Capturing_Feature_Differences_CVPR_2025_paper.html)
- [PyTorch tutorial hub](https://docs.pytorch.org/tutorials/)
- [Course roadmap](../COURSE_MAP.md)